# Waste Classification — Google Colab

Pipeline end-to-end di **disk lokal Colab** (`/content`), tanpa Google Drive.

**Sebelum mulai:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

Dataset masuk lewat unduh ZIP/RAR dari URL, lalu diextract ke `data/`.
Runtime Colab ephemeral — unduh `submission.csv` (dan checkpoint jika perlu) sebelum disconnect.

## 1. Cek GPU

In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print("device:", name)
    if "T4" not in name:
        print("Peringatan: expected T4 GPU. Pilih Runtime → Change runtime type → T4 GPU.")
    else:
        print("OK: T4 GPU siap dipakai.")
else:
    print("GPU tidak aktif. Aktifkan lewat Runtime → Change runtime type → T4 GPU.")

## 2. Clone / pull repository & install

Ganti `REPO_URL` dengan URL Git repository Anda.
- Belum ada → `git clone`
- Sudah ada → `git pull` (set `FORCE_PULL = True` untuk reset ke remote jika ada konflik lokal)

In [ ]:
from pathlib import Path

# Ganti dengan URL repository Anda
REPO_URL = "https://github.com/devkuros/RecycleLens.git"
ROOT = Path("/content/waste-python")
FORCE_PULL = False  # True = git fetch + reset --hard origin/main (buang perubahan lokal)

if not ROOT.exists():
    !git clone {REPO_URL} {ROOT}
else:
    %cd {ROOT}
    if FORCE_PULL:
        !git fetch --all
        branch = !git rev-parse --abbrev-ref HEAD
        !git reset --hard origin/{branch[0]}
        !git clean -fd
        print(f"Force pull: reset ke origin/{branch[0]}")
    else:
        !git pull --ff-only
        print(f"Pulled latest: {ROOT}")

%cd {ROOT}
!pip install -q -U pip
!pip install -q -e .

print("cwd:", Path.cwd())
!git log -1 --oneline

## 3. Siapkan dataset (disk lokal `/content`)

Archive (ZIP/RAR) harus berisi layout:

```
train/
  0_Recyclable/   # atau 0/ / Recyclable/
  1_Electronic/
  2_Organic/
test/
template.csv      # opsional jika sudah ada di repo
```

Isi `DATASET_URL` di sel bawah (link langsung atau Google Drive share), lalu jalankan.

### Unduh ZIP / RAR dari URL

Isi `DATASET_URL` (link langsung atau Google Drive share) lalu jalankan.

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path
from urllib.parse import urlparse

ROOT = Path("/content/waste-python")
DATA = ROOT / "data"
STAGING = Path("/content/_dataset_staging")
DATA.mkdir(parents=True, exist_ok=True)

# URL langsung ke .zip/.rar, atau Google Drive share link
DATASET_URL = ""  # isi URL archive dataset

def detect_archive_format(path: Path) -> str:
    """Deteksi format dari magic bytes (lebih andal daripada ekstensi)."""
    with path.open("rb") as f:
        magic = f.read(8)
    if magic.startswith(b"PK"):
        return "zip"
    if magic.startswith(b"Rar!"):
        return "rar"
    return "unknown"

def find_dataset_root(root: Path) -> Path:
    """Cari folder yang berisi train/ (hindari data/data)."""
    if (root / "train").is_dir():
        return root
    if (root / "data" / "train").is_dir():
        return root / "data"
    matches = [p.parent for p in root.rglob("train") if p.is_dir()]
    if not matches:
        raise FileNotFoundError(
            f"Tidak menemukan folder train/ di archive. Isi staging: "
            f"{sorted(p.name for p in root.iterdir())}"
        )
    return min(matches, key=lambda p: len(p.parts))

def install_from_staging(staging: Path, data_dir: Path) -> None:
    dataset_root = find_dataset_root(staging)
    data_dir.mkdir(parents=True, exist_ok=True)
    # Bersihkan data/ dulu agar tidak tersisa nest lama (data/data)
    for child in list(data_dir.iterdir()):
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    for child in dataset_root.iterdir():
        shutil.move(str(child), str(data_dir / child.name))

if not DATASET_URL:
    print("DATASET_URL kosong — isi URL archive dataset di atas.")
else:
    archive_path = Path("/content/dataset_archive")
    is_drive = any(
        host in DATASET_URL
        for host in ("drive.google.com", "docs.google.com")
    )

    if is_drive:
        !pip install -q gdown
        import gdown

        out = gdown.download(DATASET_URL, str(archive_path), quiet=False, fuzzy=True)
        if not out:
            raise RuntimeError(
                "gdown gagal mengunduh. Pastikan link Drive public/shareable."
            )
        archive_path = Path(out)
    else:
        url_name = Path(urlparse(DATASET_URL).path).name
        if Path(url_name).suffix.lower() in {".zip", ".rar"}:
            archive_path = Path("/content") / url_name
        !wget -q -O {archive_path} "{DATASET_URL}"

    if not archive_path.is_file() or archive_path.stat().st_size == 0:
        raise FileNotFoundError(f"Unduhan kosong/gagal: {archive_path}")

    if STAGING.exists():
        shutil.rmtree(STAGING)
    STAGING.mkdir(parents=True)

    fmt = detect_archive_format(archive_path)
    if fmt == "zip":
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(STAGING)
    elif fmt == "rar":
        subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=True)
        subprocess.run(
            ["unrar", "x", "-o+", str(archive_path), str(STAGING) + "/"],
            check=True,
        )
    else:
        head = archive_path.read_bytes()[:80]
        raise ValueError(
            "File yang diunduh bukan ZIP/RAR (sering karena link Drive salah "
            f"atau wget mendapat HTML). path={archive_path}, size={archive_path.stat().st_size}, "
            f"head={head!r}"
        )

    install_from_staging(STAGING, DATA)
    shutil.rmtree(STAGING, ignore_errors=True)

    print("Extracted", archive_path.name, f"({fmt}) →", DATA)
    print("isi data/:", sorted(p.name for p in DATA.iterdir()))

## 4. Verifikasi data

In [ ]:
from pathlib import Path
import sys

ROOT = Path("/content/waste-python")
sys.path.insert(0, str(ROOT / "src"))

from waste_classification.config import load_config
from waste_classification.dataset import discover_train_samples, IMAGE_EXTENSIONS

cfg = load_config(ROOT / "configs" / "default.yaml", root=ROOT)
train_dir = cfg.paths.train_dir
test_dir = cfg.paths.test_dir
template = cfg.paths.template_csv

paths, labels = discover_train_samples(train_dir)
print(f"train samples={len(paths)}")
print({c: labels.count(c) for c in (0, 1, 2)})

test_files = [
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
] if test_dir.is_dir() else []
print(f"test images={len(test_files)}")
print(f"template.csv exists={template.is_file()} → {template}")

## 5. Train

Training penuh 5-fold dengan `default.yaml` (T4 GPU Colab).

In [ ]:
%cd /content/waste-python
!python scripts/train.py --config configs/default.yaml

## 6. Predict

Pakai config yang sama dengan training di atas.

In [ ]:
%cd /content/waste-python
!python scripts/predict.py --config configs/default.yaml

## 7. Unduh hasil

Runtime Colab menghapus file saat disconnect — unduh sebelum keluar.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil

ROOT = Path("/content/waste-python")
submission = ROOT / "outputs" / "submissions" / "submission.csv"
models_dir = ROOT / "outputs" / "models"

if not submission.is_file():
    raise FileNotFoundError(f"Belum ada submission: {submission}")

files.download(str(submission))

# Opsional: unduh checkpoint
if models_dir.is_dir() and any(models_dir.glob("fold_*.pth")):
    archive = Path("/content/waste_models")
    shutil.make_archive(str(archive), "zip", models_dir)
    files.download(str(archive.with_suffix(".zip")))
    print("Downloaded submission.csv + models zip")
else:
    print("Downloaded submission.csv")